# Week 2 — Neo4j Property Graphs for Regulation

**Phase 05 · Agents & Knowledge Graphs**

This notebook demonstrates how to represent pension regulation as a **property graph** in Neo4j,
query it with Cypher, and combine graph traversal with RAG for richer answers.

## Learning objectives
1. Why property graphs outperform tables for regulatory entity relationships
2. Connect to Neo4j and verify the schema
3. Create regulation nodes and relationships with Cypher
4. Load entities extracted from `pension_regulation_excerpt.txt`
5. Query: find all articles referencing `coverage_ratio`
6. Combine graph + RAG for grounded answers
7. Walk through `neo4j_loader.py`


## 1. Why property graphs beat tables for regulation

Pension regulation is a network of references:
- Article 19 REFERENCES concept `coverage_ratio`
- `coverage_ratio` is DEFINED BY IORP III
- Requirement `coverage_ratio_minimum` REQUIRES that ABP maintain >= 100%

In a relational model, finding "all articles that reference a concept that also appears in an article
requiring a recovery plan" demands **multiple JOINs** that quickly become impractical.

In a property graph:
```cypher
MATCH (a1:Article)-[:REFERENCES]->(c:Concept)<-[:REFERENCES]-(a2:Article)-[:REQUIRES]->(r:Requirement)
WHERE r.name = 'recovery_plan_12_weeks'
RETURN a1, c, a2
```

Graph traversal is **O(degree)** rather than **O(n × m)** for JOINs on large tables.


In [ ]:
# Install dependencies (skip if already installed)
# %pip install neo4j chromadb langchain-community langchain-ollama -q

## 2. Neo4j setup and connection

In [ ]:
import os
from neo4j import GraphDatabase

NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER",     "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "pension_secret")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print(f"Connected to Neo4j at {NEO4J_URI}")

In [ ]:
def run_query(cypher: str, params: dict = None) -> list[dict]:
    """Run a Cypher query and return results as a list of dicts."""
    with driver.session() as session:
        result = session.run(cypher, **(params or {}))
        return [dict(r) for r in result]


# Quick health check
info = run_query("CALL dbms.components() YIELD name, versions RETURN name, versions")
print("Neo4j components:", info)

## 3. Creating the regulation schema — Cypher CREATE statements

### Node labels and constraints

In [ ]:
# Create uniqueness constraints (idempotent — safe to re-run)
constraints = [
    "CREATE CONSTRAINT regulation_name IF NOT EXISTS FOR (r:Regulation) REQUIRE r.name IS UNIQUE",
    "CREATE CONSTRAINT article_id      IF NOT EXISTS FOR (a:Article)    REQUIRE a.id IS UNIQUE",
    "CREATE CONSTRAINT concept_name    IF NOT EXISTS FOR (c:Concept)    REQUIRE c.name IS UNIQUE",
    "CREATE CONSTRAINT fund_name       IF NOT EXISTS FOR (f:PensionFund) REQUIRE f.name IS UNIQUE",
    "CREATE CONSTRAINT req_name        IF NOT EXISTS FOR (r:Requirement) REQUIRE r.name IS UNIQUE",
]

for cql in constraints:
    try:
        run_query(cql)
    except Exception as e:
        print(f"  (Skipped: {e})")

print("Schema constraints applied.")

In [ ]:
# Manually create a small subgraph to illustrate Cypher syntax
run_query("""
MERGE (reg:Regulation {name: 'IORP III'})
SET reg.full_name    = 'Directive on institutions for occupational retirement provision (Third)',
    reg.jurisdiction = 'European Union',
    reg.year         = 2025
""")

run_query("""
MERGE (art:Article {id: 'IORP3_ART19'})
SET art.number  = 19,
    art.title   = 'Coverage ratio and funding requirements',
    art.summary = 'IORPs must maintain coverage ratio >= 100% at all times.'
WITH art
MATCH (reg:Regulation {name: 'IORP III'})
MERGE (reg)-[:CONTAINS]->(art)
""")

run_query("""
MERGE (c:Concept {name: 'coverage_ratio'})
SET c.definition = 'Ratio of pension fund assets to technical provisions (liabilities).'
WITH c
MATCH (art:Article {id: 'IORP3_ART19'})
MERGE (art)-[:REFERENCES]->(c)
""")

print("Manual nodes created.")

## 4. Loading regulation entities from pension_regulation_excerpt.txt

We'll use the `spacy_extractor.py` module to extract entities from raw text, then insert them into Neo4j.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join("..", "knowledge_graph"))
from spacy_extractor import PensionEntityExtractor

sample_text = """
Article 19 of IORP III requires that pension funds such as ABP and PFZW maintain
a coverage ratio of at least 100% at all times. When the dekkingsgraad falls
below the minimum threshold, a recovery plan (herstelplan) must be submitted to
the supervisor within 12 weeks. The FTK additionally requires a beleidsdekkingsgraad
of at least 110% before unconditional indexation may be granted.
The prudent person principle, embedded in IORP III Article 12, mandates diversification.
The ORSA must be conducted annually under Article 14.
"""

extractor = PensionEntityExtractor()
entities = extractor.extract(sample_text)

print(f"Extracted {len(entities)} entities:")
for ent in entities:
    print(f"  [{ent['label']:15s}] '{ent['text']}'")

In [ ]:
# Insert extracted entities into Neo4j
label_to_node = {
    "PENSION_FUND": "PensionFund",
    "REGULATION":   "Regulation",
    "METRIC":       "Concept",
    "REQUIREMENT":  "Requirement",
}

for ent in entities:
    node_label = label_to_node.get(ent["label"])
    if node_label:
        run_query(
            f"MERGE (n:{node_label} {{name: $name}}) SET n.source = 'spacy_extract'",
            {"name": ent["text"]},
        )

print("Entities loaded into Neo4j.")

## 5. Graph queries — find all articles referencing coverage_ratio

In [ ]:
# Query 1: Which articles reference the coverage_ratio concept?
results = run_query("""
MATCH (a:Article)-[:REFERENCES]->(c:Concept)
WHERE c.name CONTAINS 'coverage'
RETURN a.id AS article_id, a.number AS number, a.title AS title, c.name AS concept
ORDER BY a.number
""")

print("Articles referencing coverage-related concepts:")
for r in results:
    print(f"  Art. {r['number']:>2d}  [{r['article_id']}]  {r['title']}  → concept: {r['concept']}")

In [ ]:
# Query 2: What requirements apply to a given fund?
results = run_query("""
MATCH (req:Requirement)-[:APPLIES_TO]->(f:PensionFund {name: $fund})
OPTIONAL MATCH (a:Article)-[:REQUIRES]->(req)
RETURN req.name AS requirement, req.description AS description, a.id AS article
ORDER BY req.name
""", {"fund": "ABP"})

print("Requirements for ABP:")
for r in results:
    print(f"  [{r['article'] or 'N/A'}] {r['requirement']}: {r['description']}")

In [ ]:
# Query 3: Shortest path between two concepts
results = run_query("""
MATCH p = shortestPath(
    (a:Concept {name: 'coverage_ratio'})-[*]-(b:Concept {name: 'ORSA'})
)
RETURN [n IN nodes(p) | coalesce(n.name, n.id)] AS path_nodes,
       length(p) AS hops
""")

print("Shortest path from coverage_ratio to ORSA:")
for r in results:
    print(f"  Hops: {r['hops']}  Path: {' → '.join(r['path_nodes'])}")

## 6. Combining graph + RAG — graph-guided retrieval

Strategy:
1. Use Neo4j to find the **most relevant articles** for a concept
2. Use those article IDs as filters when querying ChromaDB
3. Send the filtered chunks to the LLM for a grounded answer


In [ ]:
import chromadb
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

def graph_guided_rag(question: str, concept: str) -> str:
    """Answer a question using Neo4j to find relevant articles, then RAG for content.
    
    Args:
        question: User question.
        concept:  Key concept to start the graph traversal.
    
    Returns:
        LLM answer grounded in retrieved regulation text.
    """
    # Step 1: Graph traversal — find articles related to concept
    articles = run_query("""
    MATCH (a:Article)-[:REFERENCES]->(c:Concept)
    WHERE toLower(c.name) CONTAINS toLower($concept)
    RETURN a.id AS article_id
    """, {"concept": concept})
    
    article_ids = [r["article_id"] for r in articles]
    print(f"Graph found {len(article_ids)} relevant articles: {article_ids}")
    
    # Step 2: RAG — retrieve chunks for those articles from ChromaDB
    try:
        chroma_client = chromadb.HttpClient(host="localhost", port=8000)
        collection = chroma_client.get_collection("pension_docs")
        
        where_filter = {"article": {"$in": article_ids}} if article_ids else None
        rag_results = collection.query(
            query_texts=[question],
            n_results=3,
            where=where_filter,
            include=["documents", "metadatas"],
        )
        chunks = rag_results["documents"][0]
        context = "\n\n---\n\n".join(chunks)
    except Exception as e:
        print(f"ChromaDB unavailable ({e}), using graph summaries instead.")
        # Fallback: use article summaries from the graph
        summaries = run_query("""
        MATCH (a:Article)
        WHERE a.id IN $ids
        RETURN a.id AS id, a.title AS title, a.summary AS summary
        """, {"ids": article_ids})
        context = "\n\n".join(
            f"[{r['id']}] {r['title']}\n{r['summary']}"
            for r in summaries
        )
    
    # Step 3: LLM generates answer
    llm = ChatOllama(model="llama3.2", temperature=0.1)
    messages = [
        SystemMessage(content=(
            "You are a pension regulation expert. Answer using only the provided context. "
            "Cite article numbers where relevant."
        )),
        SystemMessage(content=f"Context:\n\n{context}"),
        HumanMessage(content=question),
    ]
    response = llm.invoke(messages)
    return response.content


# Demo
answer = graph_guided_rag(
    question="What happens when a pension fund's coverage ratio falls below 100%?",
    concept="coverage_ratio",
)
print("\nAnswer:\n", answer)

## 7. The neo4j_loader.py walkthrough

The `knowledge_graph/neo4j_loader.py` script automates the full data load pipeline:

In [ ]:
sys.path.insert(0, os.path.join("..", "knowledge_graph"))
from neo4j_loader import (
    get_driver,
    create_schema,
    load_data,
    query_related_entities,
    query_articles_for_concept,
    query_requirements_for_fund,
    REGULATIONS,
    ARTICLES,
    CONCEPTS,
)

print(f"Module defines {len(REGULATIONS)} regulations, {len(ARTICLES)} articles, {len(CONCEPTS)} concepts.")

In [ ]:
# Run the full load (safe to re-run — uses MERGE, not CREATE)
loader_driver = get_driver()
create_schema(loader_driver)
load_data(loader_driver)
print("\nFull dataset loaded.")

In [ ]:
# Use the query helper functions
print("=== Articles referencing 'coverage_ratio' ===")
for article in query_articles_for_concept("coverage_ratio", driver=loader_driver):
    print(f"  Art.{article['number']:>2d} [{article['article_id']}] — {article['title']}")

print("\n=== Requirements for PFZW ===")
for req in query_requirements_for_fund("PFZW", driver=loader_driver):
    print(f"  [{req['article_id']}] {req['requirement']}: {req['description']}")

print("\n=== Entities related to 'ORSA' ===")
for rec in query_related_entities("ORSA", driver=loader_driver)[:5]:
    print(f"  {rec['source']} --[{rec['relationship']}]--> {rec['target']} ({rec['target_type']})")

loader_driver.close()

## 8. Visualising the graph (Neo4j Browser)

Open Neo4j Browser at `http://localhost:7474` and run:

```cypher
// Show the full regulation subgraph
MATCH (reg:Regulation {name: 'IORP III'})-[:CONTAINS]->(a:Article)-[:REFERENCES]->(c:Concept)
RETURN reg, a, c
LIMIT 50
```

```cypher
// Show requirements chain for ABP
MATCH path = (reg:Regulation)-[:CONTAINS]->(a:Article)-[:REQUIRES]->(req:Requirement)-[:APPLIES_TO]->(f:PensionFund {name: 'ABP'})
RETURN path
```


## 9. Key takeaways

| Topic | Summary |
|-------|--------|
| Why graphs | Relationship traversal is native and efficient; no JOINs needed |
| MERGE | Idempotent create-or-update — safe for repeated loads |
| Cypher | Pattern matching: `(a)-[:REL]->(b)` — reads like a diagram |
| Graph + RAG | Graph narrows down relevant documents → RAG retrieves full text |
| `neo4j_loader.py` | Production loader with schema constraints, MERGE, and query helpers |

## Exercises

1. **Add new articles** — Add IORP III Articles 28 and 41 to the graph with appropriate concepts.
2. **Cross-regulation links** — Add a `SUPERSEDES` relationship from IORP III to IORP II.
3. **Fund-specific analysis** — Query which concepts appear in requirements that apply to PFZW but not ABP.
4. **Graph centrality** — Write a Cypher query that finds the concept with the most REFERENCES relationships (i.e., the most cross-referenced concept in the regulation).
5. **Full pipeline** — Modify `graph_guided_rag()` above to accept any concept dynamically and test it with `ESG` and `prudent_person_principle`.